# 🪟 10.19 Window Functions

Welcome to the second part of **SECTION F: ANALYTICAL PROCESSING**!

In **Lesson 10.15**, we learned how to use `.groupBy()` to summarize data. But `groupBy` has a major limitation: it **crushes** your data. If you have 1,000 employees and you group them by their 5 departments, your final output will only have 5 rows.

What if you want to know the average salary of a department, but you *also* want to keep all 1,000 individual employee rows on the screen to compare them against that average? 

You can't do this with a standard aggregation. You need a **Window Function**. Window Functions are the secret weapon of advanced Data Engineers and Data Analysts. They allow you to look at a "window" of surrounding rows without crushing the underlying data.

### 🎯 Learning Objectives
* Understand the difference between `groupBy` and **Window Functions**.
* Learn how to define a window using `Window.partitionBy()` and `orderBy()`.
* Master ranking data using `row_number()`, `rank()`, and `dense_rank()`.
* Master time-travel analytics (like Month-over-Month growth) using `lead()` and `lag()`.
* See how mastering analytical functions sets Senior Data Engineers apart.

In [ ]:
# 0. Setup: Let's start our SparkSession and create a small, competitive Sales team dataset.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Window_Functions").master("local[*]").getOrCreate()

data = [
    ("Alice", "Sales", 100000),
    ("Bob", "Sales", 80000),
    ("Charlie", "Sales", 80000),   # Bob and Charlie have the exact same sales!
    ("Diana", "Sales", 60000),
    ("Eve", "Engineering", 95000),
    ("Frank", "Engineering", 90000)
]
columns = ["emp_name", "department", "revenue"]

df = spark.createDataFrame(data, columns)
print("Raw Employee Revenue Data:")
df.show()

## 1. GroupBy vs. Window Functions

Let's visualize the difference.

* **`groupBy()` (The Hydraulic Press):** Takes 100 rows, squashes them together into 1 row, and outputs the sum. The individual rows are lost forever.
* **`Window` (The Magnifying Glass):** Looks at row #1, peeks at the rows around it, calculates the sum, and writes the answer in a *new column* right next to row #1. It repeats this for row #2. **No rows are lost!**

<img src="../assets/Module_10/10_19_01.png" alt="A visual comparison. Left side shows a hydraulic press crushing a stack of papers into a single brick (groupBy). Right side shows a magnifying glass sliding down a stack of papers, writing a note on each paper without destroying them (Window Function)." width="75%">

## 2. Defining the Window: `partitionBy` and `orderBy`

Before we can use a Window Function, we have to tell Spark the size and shape of our "Window". We do this using the `Window` class.

1. **`partitionBy()`:** Tells Spark how to group the rows (just like `groupBy`). 
2. **`orderBy()`:** Tells Spark how to sort the rows *inside* that window.

Let's create a Window that looks at employees department-by-department, ordered from highest revenue to lowest.

In [ ]:
# We must import the Window class to define our boundaries
from pyspark.sql.window import Window

# Defining the Window Specification
# "Group everyone by department, and inside that department, line them up by revenue (highest first)"
window_spec = Window.partitionBy("department").orderBy(F.desc("revenue"))

print("Window Specification successfully created!")
print("(Because of Lazy Evaluation, nothing happens yet.)")

## 3. Ranking Functions: `row_number`, `rank`, and `dense_rank`

Now that our window is defined, let's figure out who is the #1 salesperson in each department! 

There are three different ways to rank data. Let's apply all three using `.withColumn()` and passing our `window_spec` into the `.over()` method.

<img src="../assets/Module_10/10_19_02.png" alt="A podium showing 1st, 2nd, and 3rd place. Next to it is a table explaining that if there is a tie for 2nd, Rank skips 3rd place (1, 2, 2, 4), while Dense Rank does not skip (1, 2, 2, 3)." width="75%">

In [ ]:
print("--- Ranking Employees by Department ---")

df_ranked = df \
    .withColumn("row_number", F.row_number().over(window_spec)) \
    .withColumn("rank", F.rank().over(window_spec)) \
    .withColumn("dense_rank", F.dense_rank().over(window_spec))

df_ranked.show()

print("Look closely at the Sales department! Bob and Charlie tied at $80,000.")
print(" - row_number: Just counts 1, 2, 3, 4 blindly. (Bob is 2, Charlie is 3).")
print(" - rank: Gives them both 2nd place, but SKIPS 3rd place! (Diana gets 4th).")
print(" - dense_rank: Gives them both 2nd place, and DOES NOT skip. (Diana gets 3rd).")

## 4. Time Travel: `lead()` and `lag()`

Ranking is great, but Window functions have another superpower: **Looking into the past and future**.

In traditional Python or SQL, comparing Row 2 to Row 1 is incredibly difficult because rows aren't supposed to "know" about each other.

* **`lag(column, offset)`**: Looks *backwards* (up) a certain number of rows.
* **`lead(column, offset)`**: Looks *forwards* (down) a certain number of rows.

Let's create a quick dataset of monthly sales to see how Data Engineers calculate "Month-over-Month Growth."

In [ ]:
print("--- Calculating Month-Over-Month Growth using LAG ---")

# 1. Create a Time-Series Dataset
time_data = [
    ("Store_A", "2023-01", 100),
    ("Store_A", "2023-02", 150),
    ("Store_A", "2023-03", 130)
]
time_df = spark.createDataFrame(time_data, ["store", "month", "sales"])

# 2. Define a time-based window (Sort by month chronologically)
time_window = Window.partitionBy("store").orderBy("month")

# 3. Use LAG to pull the previous month's sales onto the CURRENT row
df_growth = time_df \
    .withColumn("previous_month_sales", F.lag("sales", 1).over(time_window)) \
    .withColumn("difference", F.col("sales") - F.col("previous_month_sales"))

df_growth.show()

print("Notice that January has a 'null' previous month because it is the first row in the window!")

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Window Functions are where basic SQL ends and advanced analytics begins.

| Feature | 🏛️ Data Analyst | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Use of Window Functions** | Writes them in SQL (`SUM(sales) OVER (PARTITION BY...)`) to create BI dashboards. | **Writes them in PySpark. Uses them heavily to deduplicate data (e.g., keeping only the most recent row `row_number() == 1`).** | Uses them to engineer features for AI models (e.g., calculating a 7-day rolling average of a stock price). |
| **Performance Awareness** | Runs the query and hopes it finishes quickly. | **Knows that `Window.partitionBy()` causes a Network Shuffle (just like `groupBy()`) and manages the partitions so the cluster doesn't choke.** | Ignores the cluster; just wants the final features. |
| **Syntax Preference** | Pure SQL. | **PySpark Method Chaining (more dynamic and programmable).** | Pandas (`df.shift()` for lag). |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Relies entirely on `groupBy`. If asked to calculate Month-over-Month growth, they will write a highly complex, inefficient `JOIN` (joining the table to itself offset by one month).
2. **Mid-Level DE (Your Current Phase!):** Adopts **Window Functions**. Understands how to use `lag()` to replace complex self-joins, making the code 10x faster and infinitely easier to read.
3. **Senior DE:** Masters **Rolling Windows**. A Senior DE knows how to define an advanced window that says: *"Don't just partition the data, but specifically calculate the sum of sales for only the preceding 7 days relative to the current row."* (Using `rowsBetween` and `rangeBetween`).

### 🛠️ Where Data Engineering Fits in Modern Organizations
When an executive opens a dashboard and sees "You are the #3 salesperson this month!" or "Sales are up 15% from yesterday," they are looking at the output of a Data Engineer's Window Function pipeline.

## 🔑 Key Takeaways

- **Window Functions** allow you to calculate aggregations or rankings across a group of rows *without* collapsing those rows into a single summary output.
- **`Window.partitionBy().orderBy()`**: The required syntax to tell Spark the boundaries of your window and how to sort the data inside it.
- **Ranking**: `row_number()` blindly numbers rows 1,2,3. `rank()` handles ties but skips numbers. `dense_rank()` handles ties but never skips numbers.
- **`lag()` and `lead()`**: Allows a row to "peek" at the values of the rows immediately preceding or following it. Essential for time-series math.
- Like `groupBy()`, using `partitionBy()` triggers a **Network Shuffle**, which is an expensive operation on a cluster.

## 📝 Knowledge Check Quiz

**Question 1:** Why would a Data Engineer use a Window Function to calculate an average, rather than just using a standard `groupBy().agg()` function?
A) Because Window Functions do not require a network shuffle.
B) Because `groupBy` crushes the individual rows into a single summary row, whereas a Window Function appends the average as a new column while keeping all the original, individual rows intact.
C) Because Window Functions are the only way to do math in Spark.
D) Because `groupBy` only works on numbers.

**Question 2:** Two employees in your dataset have the exact same highest sales amount ($100,000). You apply a ranking function. They both get rank `1`. The person with the next highest sales ($90,000) gets rank `2` (the number 2 was NOT skipped). Which function did you use?
A) `row_number()`
B) `rank()`
C) `dense_rank()`
D) `lead()`

**Question 3:** Which Window function is perfectly designed to help you calculate the day-over-day change in a stock's price?
A) `row_number()`
B) `lag()`
C) `dense_rank()`
D) `count()`

---

*Answers: 1: B, 2: C, 3: B*

In [ ]:
# Clean up our session
spark.stop()
print("Spark session closed.")

### 🚀 What's Next?
You now have a complete toolkit for analytical processing. You can filter, shape, aggregate, and rank data.

But what if you need to calculate something so completely unique to your business that Spark doesn't have a built-in function for it? 

In **SECTION G: CUSTOM LOGIC**, we will learn how to write our own custom Python functions and force Spark to run them across the cluster. See you in **10.20 Built-in Functions vs UDFs**!